데이터, 라이브러리 호출

In [ ]:
import pandas as pd
import ast
import numpy as np

df_reviews_raw = pd.read_csv('steam_indie_reviews.csv')
df_sample_raw = pd.read_csv('steam_stratified_sample.csv')

C:\Users\asdxo\AppData\Local\Temp\ipykernel_20868\851431824.py:5: DtypeWarning: Columns (0: early_access_review) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv('merged_steam_reviews.csv')


In [ ]:
df_reviews = df_reviews_raw.copy()
df_sample = df_sample_raw.copy()

# appid를 무조건 문자열(str)로 통일
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

# 2. 분석 대상(74개) 필터링
target_strata = ['large_high', 'mid_high', 'small_high']
df_sample_filtered = df_sample[df_sample['stratum'].isin(target_strata)].copy()

# 3. 날짜 데이터 변환
df_sample_filtered['release_date_dt'] = pd.to_datetime(df_sample_filtered['release_date'], errors='coerce').dt.tz_localize(None)

if 'timestamp_created' in df_reviews.columns:
    df_reviews['review_date_dt'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s', errors='coerce').dt.tz_localize(None)
else:
    df_reviews['review_date_dt'] = pd.to_datetime(df_reviews['post_date'], errors='coerce').dt.tz_localize(None)

# 4. 추천 여부를 알 수 있는 컬럼 찾기
possible_cols = ['voted_up', 'recommend']
target_col = None

for col in possible_cols:
    if col in df_reviews.columns:
        target_col = col
        break

if target_col is None:
    print(f"에러: 추천 여부를 알 수 있는 컬럼({possible_cols})을 찾을 수 없습니다.")
else:
    print(f"사용된 추천 컬럼: '{target_col}'")

    # 5. 데이터 병합 (74개 전부를 살리기 위해 Left Join 사용)
    df_merged = pd.merge(
        df_sample_filtered[['appid', 'name_store', 'release_date_dt', 'total_reviews', 'stratum']], 
        df_reviews[['appid', target_col, 'review_date_dt']], 
        on='appid', 
        how='left'
    )

    # 6. 출시 후 90일 이내 리뷰 필터링
    df_merged['days_since_release'] = (df_merged['review_date_dt'] - df_merged['release_date_dt']).dt.days
    
    df_early = df_merged[(df_merged['days_since_release'].isna()) | 
                         ((df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90))].copy()

    # ==========================================
    # 7. 긍정률 계산 (에러 해결: float로 변환하여 NaN 허용)
    # ==========================================
    df_early['is_positive'] = df_early[target_col].astype(str).str.lower().isin(['true', '1', 'recommended', 'yes']).astype(float)
    df_early.loc[df_early[target_col].isna(), 'is_positive'] = np.nan 

    # 8. 게임별 집계
    early_summary = df_early.groupby(['appid', 'name_store', 'stratum', 'total_reviews']).agg(
        early_review_count=(target_col, 'count'), 
        early_positive_rate=('is_positive', lambda x: (x.sum() / x.count()) * 100 if x.count() > 0 else 0)
    ).reset_index()

    final_comparison = early_summary.sort_values(by='early_review_count', ascending=False)

    print(f"\n🎯 [최종 확인] {len(final_comparison)}개 게임의 초동 성적 (목표 74개 달성!)")
    display(final_comparison.head(10))

    # 9. 상관관계 계산
    corr_spearman = final_comparison['early_review_count'].corr(final_comparison['total_reviews'], method='spearman')
    print(f"\n📊 74개 타겟 그룹 스피어만 상관계수: {corr_spearman:.3f}")

그럼 장르별 상관관계는?

In [ ]:
# 1. 리뷰 데이터와 샘플 데이터 병합 (장르 분석을 위해 inner join 사용)
df_merged_genre = pd.merge(
    df_reviews, 
    df_sample_filtered[['appid', 'name_store', 'release_date_dt', 'genres', 'total_reviews']], 
    on='appid', 
    how='inner' 
)

# 2. 출시 후 90일 이내 리뷰 필터링
EARLY_DAYS = 90
df_merged_genre['days_since_release'] = (df_merged_genre['review_date_dt'] - df_merged_genre['release_date_dt']).dt.days
df_early_genre = df_merged_genre[(df_merged_genre['days_since_release'] >= 0) & (df_merged_genre['days_since_release'] <= EARLY_DAYS)].copy()

# 3. 장르(genres) 텍스트를 리스트로 변환 후 분리 (Explode)
def parse_genres(genre_str):
    try: return ast.literal_eval(genre_str)
    except: return []

df_early_genre['genre_list'] = df_early_genre['genres'].apply(parse_genres)
df_exploded = df_early_genre.explode('genre_list')

# 'Indie' 장르는 공통이므로 분석에서 제외
df_exploded = df_exploded[df_exploded['genre_list'] != 'Indie']

# 4. 장르별 + 게임별 초기 성적 집계
game_genre_summary = df_exploded.groupby(['genre_list', 'appid', 'name_store']).agg(
    early_review_count=('appid', 'count'),
    final_total_reviews=('total_reviews', 'first')
).reset_index()

# 5. [핵심] 장르별로 상관계수 계산
def calculate_genre_corr(group):
    # 해당 장르의 게임이 너무 적으면(예: 5개 미만) 상관계수 신뢰도가 낮아 제외
    if len(group) < 5: 
        return None
    return group['early_review_count'].corr(group['final_total_reviews'], method='spearman')

genre_correlations = game_genre_summary.groupby('genre_list').apply(calculate_genre_corr).reset_index()
genre_correlations.columns = ['Genre', 'Spearman_Correlation']

# 결과 정렬 (상관계수 높은 순)
genre_correlations = genre_correlations.dropna().sort_values(by='Spearman_Correlation', ascending=False)

print(f"\n🎯 [장르별] 초기 90일 리뷰 수 vs 최종 흥행 상관관계")
display(genre_correlations.round(3))

KeyError: 'timestamp_created'